In [ ]:
pip install vaderSentiment

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import requests
from bs4 import BeautifulSoup
import re
import json

import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
# =========================
# 2. LOAD NLP MODELS
# =========================
nlp = spacy.load("en_core_web_sm")
sentiment_analyzer = SentimentIntensityAnalyzer()

In [ ]:
# =========================
# 3. SCRAPE MONEYCONTROL
# =========================
def scrape_moneycontrol():
    url = "https://www.moneycontrol.com/news/business/"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    articles = soup.find_all("li", class_="clearfix")
    news_list = []

    for article in articles[:5]:
        title = article.find("h2").text.strip()
        link = article.find("a")["href"]

        news_list.append({
            "source": "Moneycontrol",
            "headline": title,
            "url": link
        })

    return news_list

In [ ]:
# =========================
# 4. TEXT CLEANING
# =========================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

In [ ]:
# =========================
# 5. NER EXTRACTION
# =========================
def extract_entities(text):
    doc = nlp(text)
    entities = {"ORG": [], "GPE": [], "FACILITY": []}

    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)

    return entities

In [ ]:
# =========================
# 6. KEYWORD & EVENT EXTRACTION
# =========================
def extract_event(text):
    keywords = ["fire", "explosion", "accident", "blast", "halt", "shutdown"]
    detected = [kw for kw in keywords if kw in text]

    return detected

In [ ]:
# =========================
# 7. INDUSTRY CLASSIFICATION
# =========================
def classify_industry(text):
    pharma_words = ["pharma", "drug", "medicine", "plant", "laboratory"]

    for word in pharma_words:
        if word in text:
            return "Pharmaceutical"

    return "Other"

In [ ]:
def sentiment_analysis(text):
    positive_words = [
        "growth", "profit", "success", "gain", "positive",
        "improve", "strong", "benefit", "increase", "record"
    ]

    negative_words = [
        "loss", "fire", "accident", "shutdown", "halt",
        "decline", "drop", "risk", "failure", "damage"
    ]

    words = text.lower().split()

    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)

    # Manual sentiment score
    score = (pos_count - neg_count) / (len(words) + 1)

    if score <= -0.05:
        sentiment = "Negative"
    elif score >= 0.05:
        sentiment = "Positive"
    else:
        sentiment = "Neutral"

    return sentiment, round(score, 3)


In [ ]:
# =========================
# 8. SENTIMENT ANALYSIS
# =========================
# def sentiment_analysis(text):
#     score = sentiment_analyzer.polarity_scores(text)["compound"]

#     if score <= -0.6:
#         sentiment = "Negative"
#     elif score >= 0.6:
#         sentiment = "Positive"
#     else:
#         sentiment = "Neutral"

#     return sentiment, score

In [ ]:
# =========================
# SEVERITY MODEL
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# -------------------------
# 1. TRAINING DATA
# -------------------------
train_data = [
    ("fire accident at manufacturing plant production halted", "High"),
    ("explosion damages factory operations stopped", "High"),
    ("chemical leak causes temporary shutdown", "High"),

    ("regulatory probe causes operational disruption", "Medium"),
    ("supply chain delay affects quarterly output", "Medium"),
    ("maintenance issue slows production", "Medium"),

    ("company reports marginal decline in profit", "Low"),
    ("stock rises after strong earnings", "Low"),
    ("new product launch boosts revenue", "Low"),
]

texts = [t[0] for t in train_data]
labels = [t[1] for t in train_data]

# -------------------------
# 2. VECTORIZE TEXT
# -------------------------
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(texts)

# -------------------------
# 3. TRAIN MODEL
# -------------------------
severity_model = LogisticRegression(max_iter=1000)
severity_model.fit(X_train, labels)

# -------------------------
# 4. PREDICTION FUNCTION
# -------------------------
def classify_severity(text):
    X_test = vectorizer.transform([text.lower()])
    severity = severity_model.predict(X_test)[0]
    confidence = max(severity_model.predict_proba(X_test)[0])
    return severity, round(confidence, 2)

# -------------------------
# 5. TEST
# -------------------------
sample_news = "Fire accident at pharma manufacturing plant halts production"
print(classify_severity(sample_news))


(np.str_('High'), np.float64(0.45))


In [ ]:
# # =========================
# # 9. SEVERITY CLASSIFICATION
# # =========================
# def classify_severity(event_keywords, sentiment_score):
#     if any(k in event_keywords for k in ["fire", "explosion", "accident"]) and sentiment_score < -0.6:
#         return "High", 0.91
#     elif sentiment_score < -0.3:
#         return "Medium", 0.70
#     else:
#         return "Low", 0.40

In [ ]:
# =========================
# 10. PIPELINE EXECUTION
# =========================
def run_pipeline():
    news = scrape_moneycontrol()
    results = []

    for item in news:
        raw_text = item["headline"]
        clean = clean_text(raw_text)

        entities = extract_entities(raw_text)
        industry = classify_industry(clean)
        sentiment, score = sentiment_analysis(clean)

        # ✅ ML-based severity (TEXT only)
        severity, confidence = classify_severity(clean)

        output = {
            "source": item["source"],
            "headline": raw_text,
            "company": entities.get("ORG"),
            "location": entities.get("GPE"),
            "industry": industry,
            "sentiment": sentiment,
            "sentiment_score": score,
            "severity": severity,
            "confidence": confidence
        }

        results.append(output)

    return results


In [ ]:
# =========================
# 11. RUN
# =========================
if __name__ == "__main__":
    output = run_pipeline()
    print(json.dumps(output, indent=4))

[
    {
        "source": "Moneycontrol",
        "headline": "Buy GE Vernova TD India; target of Rs 4005: Prabhudas Lilladher",
        "company": [
            "GE",
            "TD India"
        ],
        "location": [],
        "industry": "Other",
        "sentiment": "Neutral",
        "sentiment_score": 0.0,
        "severity": "Low",
        "confidence": 0.34
    },
    {
        "source": "Moneycontrol",
        "headline": "OPINION | Reimagining wealth management,\u00a0a digital-first future \u200b",
        "company": [],
        "location": [],
        "industry": "Other",
        "sentiment": "Neutral",
        "sentiment_score": 0.0,
        "severity": "Low",
        "confidence": 0.34
    },
    {
        "source": "Moneycontrol",
        "headline": "GDP numbers unlikely to change much with new GDP series: MoSPI secretary Saurabh Garg",
        "company": [],
        "location": [],
        "industry": "Other",
        "sentiment": "Neutral",
        "sentiment_scor